# Agent Zero — Fixed Colab Setup

This notebook installs and launches [Agent Zero](https://github.com/agent0ai/agent-zero) in Google Colab
and exposes it to the browser via an ngrok tunnel.

**Root cause of the previous `Internal Server Error`:**
The original setup tried to pin `openai==1.50.2` together with `litellm==1.49.5`.
`litellm 1.49.5` internally imports `PromptTokensDetails` from
`openai.types.completion_usage`, a class that does *not* exist in `openai 1.50.2`
(only `CompletionTokensDetails` is present in that release).  
The fix is to let pip install a compatible set of modern versions instead of forcing
two incompatible old releases.

## Step 1 — Clone Agent Zero and upgrade build tools

In [ ]:
%cd /content
!rm -rf agent-zero
!git clone https://github.com/agent0ai/agent-zero.git
%cd agent-zero

!pip install --upgrade pip setuptools wheel -q

## Step 2 — Diagnose the current environment

In [ ]:
import importlib.metadata
import os

libs = ['openai', 'litellm', 'pydantic', 'typing-extensions', 'anyio']
print('--- Installed versions ---')
for lib in libs:
    try:
        print(f"{lib}: {importlib.metadata.version(lib)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{lib}: NOT INSTALLED")

print('\n--- Working directory ---')
print(os.getcwd())

print('\n--- .env presence ---')
print(f"root .env : {os.path.exists('.env')}")
print(f"usr/.env  : {os.path.exists('usr/.env')}")

## Step 3 — Install compatible dependencies

Instead of pinning the two conflicting old releases, we install the latest
compatible versions and let pip resolve the dependency graph correctly.

In [ ]:
# Purge the pip cache so no stale wheels can interfere.
!pip cache purge

# Install Agent Zero's requirements from its own file first.
!pip install -r requirements.txt -q

# Ensure the web-server and tunnel dependencies are present.
!pip install flask flask-cors pydantic python-dotenv pyngrok -q

## Step 4 — Verify the installation

In [ ]:
import importlib
import importlib.metadata

errors = []
for pkg in ('openai', 'litellm', 'flask', 'pydantic'):
    try:
        mod = importlib.import_module(pkg)
        ver = importlib.metadata.version(pkg)
        print(f"\u2705 {pkg} {ver}")
    except Exception as exc:
        print(f"\u274c {pkg} — {exc}")
        errors.append(pkg)

# Verify that the specific attribute that previously caused the crash is accessible.
try:
    import litellm  # noqa: F401
    print('\u2705 litellm imported cleanly (no PromptTokensDetails error)')
except Exception as exc:
    print(f'\u274c litellm import failed: {exc}')
    errors.append('litellm-import')

if not errors:
    print('\n\u2705 All packages verified. Ready to launch Agent Zero.')
else:
    print(f'\n\u26a0\ufe0f Issues found in: {errors}. See troubleshooting notes below.')

## Step 5 — Configure environment variables

Edit the cell below to add your API key(s) before running it.

In [ ]:
import os

# ── Set your API keys here ──────────────────────────────────────────────────
ANTHROPIC_API_KEY = ""   # e.g. "sk-ant-..."
OPENAI_API_KEY    = ""   # e.g. "sk-..."
NGROK_AUTH_TOKEN  = ""   # from https://dashboard.ngrok.com/get-started/your-authtoken
# ───────────────────────────────────────────────────────────────────────────

env_lines = []
if ANTHROPIC_API_KEY:
    env_lines.append(f"ANTHROPIC_API_KEY={ANTHROPIC_API_KEY}")
if OPENAI_API_KEY:
    env_lines.append(f"OPENAI_API_KEY={OPENAI_API_KEY}")

# Write to the location Agent Zero expects.
os.makedirs('usr', exist_ok=True)
with open('usr/.env', 'w') as fh:
    fh.write('\n'.join(env_lines) + '\n')

print(f"usr/.env written with {len(env_lines)} key(s).")

## Step 6 — Launch Agent Zero and expose it via ngrok

In [ ]:
import subprocess
import time
import requests
from pyngrok import ngrok, conf

PORT = 5000

# ── Configure ngrok ────────────────────────────────────────────────────────
if not NGROK_AUTH_TOKEN:
    raise ValueError("Set NGROK_AUTH_TOKEN in Step 5 before running this cell.")

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# ── Start the Agent Zero web server ───────────────────────────────────────
server_proc = subprocess.Popen(
    ["python", "run_ui.py", "--port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("Agent Zero process started (PID", server_proc.pid, ")")

# ── Wait until the server is ready ────────────────────────────────────────
print("Waiting for server to become ready ", end="")
for _ in range(30):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        if requests.get(f"http://localhost:{PORT}", timeout=2).status_code < 500:
            print(" ready!")
            break
    except Exception:
        pass
else:
    print(" timed out — check server logs above.")

# ── Open ngrok tunnel ─────────────────────────────────────────────────────
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print("=" * 60)
print("\U0001f680 AGENT ZERO IS LIVE!")
print(f"\U0001f517 URL: {public_url}")
print("=" * 60)
print("Open the URL above in your browser to use the Agent Zero UI.")

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `cannot import name 'PromptTokensDetails'` | `litellm` version < 1.50 expects an attribute absent in `openai` < 1.51 | Re-run **Step 3** — the modern versions are compatible |
| `ResolutionImpossible` from pip | Pinning `openai==1.50.2` + `litellm==1.49.5` together | Do **not** pin those old versions; let pip choose |
| `Internal Server Error` in the UI | Stale `.pyc` / cached wheels from a previous run | Re-run `pip cache purge` in **Step 3** |
| ngrok `ERR_NGROK_108` | Auth token missing or expired | Set `NGROK_AUTH_TOKEN` in **Step 5** |
| Server never becomes ready | Flask failed to start | Check `server_proc.stdout` for Python tracebacks |